# Amazon SageMaker — Simple ML Training Demo

Flow: **CSV → S3 → SageMaker Training Job → model.tar.gz → S3 → CloudWatch Logs**


In [ ]:
import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

session = sagemaker.Session()
region = session.boto_region_name

try:
    role = sagemaker.get_execution_role()
except Exception:
    role = "arn:aws:iam::<ACCOUNT_ID>:role/<SAGEMAKER_EXECUTION_ROLE>"

bucket = session.default_bucket()
prefix = "simple-ml-demo"

print("Region:", region)
print("S3 bucket:", bucket)
print("Role:", role)


In [ ]:
# 1. Upload the CSV dataset to S3
local_csv = "data/houses.csv"

s3_train_uri = session.upload_data(
    path=local_csv,
    bucket=bucket,
    key_prefix=f"{prefix}/input/train"
)

print("Training data:", s3_train_uri)


In [ ]:
# 2. Define the SageMaker scikit-learn training job
sklearn_estimator = SKLearn(
    entry_point="train.py",
    source_dir="code",
    role=role,
    instance_type="ml.m5.large",
    framework_version="1.4-2",
    py_version="py3",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=session,
    base_job_name="simple-house-price"
)

print("Estimator created")


In [ ]:
# 3. Submit the training job
# 'train' becomes the input channel name inside the training container.
sklearn_estimator.fit({"train": s3_train_uri}, wait=True)

print("Training completed!")
print("Training job:", sklearn_estimator.latest_training_job.name)
print("Model artifact:", sklearn_estimator.model_data)


In [ ]:
# 4. List the S3 objects created by this demo
s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

for obj in response.get("Contents", []):
    print(obj["Key"])


In [ ]:
# 5. Optional: download the model artifact and inspect its contents
# Run these in a notebook cell if you want to inspect model.tar.gz locally:
# !aws s3 cp "{sklearn_estimator.model_data}" ./model.tar.gz
# !tar -tzf ./model.tar.gz


## What to review in the AWS console

- **S3**: `simple-ml-demo/input/train/` contains the training CSV.
- **SageMaker → Training jobs**: open the generated training job.
- **Logs**: review the printed row count, MAE, R² and model coefficients.
- **Outputs**: open the S3 model artifact location; SageMaker stores the trained output as `model.tar.gz`.
- **CloudWatch Logs**: training job logs are available under the SageMaker training log group.
